# FedUCA Phase 3: Prototype-Guided Aggregation (PGA)

**Add data:** `ham10000-vit-embeddings-orig` + `feduca-phase2-results` | **GPU T4** | Internet OFF

In [ ]:
import os, json, time, copy, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_phase3"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7
EMBED_DIM   = 768

# Auto-find embeddings
hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("embeddings_train.npz not found. Add ham10000-vit-embeddings-orig dataset.")
EMBED_DIR = os.path.dirname(hits[0])

# Auto-find Phase 2 results
p2_hits = glob.glob("/kaggle/input/**/fedavg_bottleneck_r4_a1000_s0.json", recursive=True)
if not p2_hits:
    raise FileNotFoundError("Phase 2 results not found. Add feduca-phase2-results dataset.")
P2_DIR = os.path.dirname(p2_hits[0])

print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")
print(f"P2 results: {P2_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    return (torch.tensor(d['embeddings'], dtype=torch.float32),
            torch.tensor(d['labels'],     dtype=torch.long))

emb_train, lbl_train = load_split("train")
emb_val,   lbl_val   = load_split("val")
emb_test,  lbl_test  = load_split("test")

print(f"Train: {emb_train.shape} | Val: {emb_val.shape} | Test: {emb_test.shape}")

counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = 1.0 / counts
weights = (weights / weights.mean()).to(DEVICE)
print(f"Class weights: {weights.cpu().numpy().round(3)}")

In [ ]:
# Shared model + utilities
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768, 4*b), nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, b),   nn.GELU(), nn.LayerNorm(b),   nn.Dropout(0.2),
            nn.Linear(b, 4*b),   nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, 7))
    def forward(self, x): return self.net(x)

def dirichlet_partition(labels, K, alpha, seed):
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max()) + 1):
        ci = np.where(labels.numpy() == c)[0]
        rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props) * len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval()
    p, t = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu())
        t.append(y)
    p = torch.cat(p).numpy()
    t = torch.cat(t).numpy()
    f1s = []
    for c in range(NUM_CLASSES):
        tp = ((p==c)&(t==c)).sum(); fp = ((p==c)&(t!=c)).sum(); fn = ((p!=c)&(t==c)).sum()
        pr = tp/(tp+fp+1e-9);       rc = tp/(tp+fn+1e-9)
        f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy": float((p==t).mean()), "macro_f1": float(np.mean(f1s))}

def local_train(head, embs, labels, lr=5e-3, epochs=5, bs=32):
    h    = copy.deepcopy(head).to(DEVICE); h.train()
    opt  = optim.Adam(h.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        for x, y in DataLoader(TensorDataset(embs.to(DEVICE), labels.to(DEVICE)),
                               batch_size=bs, shuffle=True):
            opt.zero_grad(); crit(h(x), y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def fedavg(states, sizes):
    total = sum(sizes)
    return {k: sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

print("Utilities ready.")

In [ ]:
# ── NOVEL CONTRIBUTION: Prototype-Guided Aggregation ─────────────────────────

def compute_prototypes(embs, labels):
    proto = torch.zeros(NUM_CLASSES, EMBED_DIM)
    count = torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask = labels == c
        if mask.sum() > 0:
            proto[c] = embs[mask].mean(0)
            count[c] = float(mask.sum())
    return proto, count   # (7, 768), (7,)

def prototype_guided_aggregation(states, sizes, client_protos, beta=5.0):
    K     = len(states)
    total = sum(sizes)

    # Global prototype consensus (size-weighted)
    g_proto = torch.zeros(NUM_CLASSES, EMBED_DIM)
    g_count = torch.zeros(NUM_CLASSES)
    for k in range(K):
        proto_k, count_k = client_protos[k]
        for c in range(NUM_CLASSES):
            if count_k[c] > 0:
                g_proto[c] += count_k[c] * proto_k[c]
                g_count[c] += count_k[c]
    for c in range(NUM_CLASSES):
        if g_count[c] > 0:
            g_proto[c] /= g_count[c]

    # Per-class softmax weights over clients
    def cos_dist(a, b):
        na = a / (a.norm() + 1e-9)
        nb = b / (b.norm() + 1e-9)
        return 1.0 - float((na * nb).sum())

    w = torch.zeros(NUM_CLASSES, K)
    for c in range(NUM_CLASSES):
        if g_proto[c].norm() < 1e-9:
            for k in range(K):
                w[c, k] = sizes[k] / total
        else:
            lw = torch.zeros(K)
            for k in range(K):
                pk, ck = client_protos[k]
                lw[k] = -beta * cos_dist(pk[c], g_proto[c]) if ck[c] > 0 else -1e6
            w[c] = torch.softmax(lw, dim=0)

    # Find final layer keys (weight shape [7,*] and bias shape [7])
    fw_key = fb_key = None
    for key in states[0]:
        shape = states[0][key].shape
        if len(shape) == 2 and shape[0] == NUM_CLASSES:
            fw_key = key
        if len(shape) == 1 and shape[0] == NUM_CLASSES:
            fb_key = key

    avg = {}
    for key in states[0]:
        if key == fw_key:
            avg[key] = torch.stack([
                sum(w[c, k] * states[k][key][c].float() for k in range(K))
                for c in range(NUM_CLASSES)])
        elif key == fb_key:
            avg[key] = torch.stack([
                sum(w[c, k] * states[k][key][c].float() for k in range(K))
                for c in range(NUM_CLASSES)])
        else:
            avg[key] = sum((sizes[i]/total) * states[i][key].float()
                          for i in range(K))
    return avg

overhead_kb = NUM_CLASSES * EMBED_DIM * 4 / 1024
print(f"Prototype overhead per round: {overhead_kb:.1f} KB (negligible)")
print("PGA ready.")

In [ ]:
# PGA training loop
def run_pga(alpha, seed, beta=5.0, n_rounds=60, K=4, epochs=5, lr=5e-3, bs=32):
    partition = dirichlet_partition(lbl_train, K, alpha, seed)
    c_embs    = [emb_train[idx] for idx in partition]
    c_labels  = [lbl_train[idx] for idx in partition]
    sizes     = [len(idx)       for idx in partition]

    # Prototypes from frozen embeddings — computed once, never changes
    c_protos = [compute_prototypes(c_embs[k], c_labels[k]) for k in range(K)]

    head    = BottleneckMLP(ratio=4).to(DEVICE)
    history = []; best_val = -1; best_round = 0; best_state = None
    t0 = time.time()

    for rnd in range(1, n_rounds + 1):
        states = [local_train(head, c_embs[k], c_labels[k], lr, epochs, bs)
                  for k in range(K)]
        head.load_state_dict(
            prototype_guided_aggregation(states, sizes, c_protos, beta))
        vm = evaluate(head, emb_val.to(DEVICE), lbl_val)
        history.append(vm['macro_f1'])
        if vm['macro_f1'] > best_val:
            best_val = vm['macro_f1']; best_round = rnd
            best_state = copy.deepcopy(head.state_dict())
        if rnd % 10 == 0 or rnd == 1:
            print(f"  [PGA] r{rnd:3d} val_F1={vm['macro_f1']:.4f} "
                  f"best={best_val:.4f}@r{best_round} | {time.time()-t0:.0f}s")

    head.load_state_dict(best_state)
    tm = evaluate(head, emb_test.to(DEVICE), lbl_test)
    return {"method": "PGA", "alpha": alpha, "seed": seed, "beta": beta,
            "best_round": best_round, "best_val_f1": best_val,
            "val_history": history, "test": tm}

print("run_pga() ready.")

In [ ]:
# Run PGA across all alpha values and seeds
print("="*55)
print("PHASE 3: PGA vs FedAvg (Bottleneck N/4, 5 seeds x 3 alpha)")
print("="*55)

ALPHAS = [1000, 1, 0.1]
SEEDS  = [0, 1, 2, 3, 4]
pga_results = []

for alpha in ALPHAS:
    print(f"\n=== alpha={alpha} ===")
    for seed in SEEDS:
        print(f"  PGA alpha={alpha} seed={seed}")
        res = run_pga(alpha=alpha, seed=seed)
        pga_results.append(res)
        with open(os.path.join(RESULTS_DIR, f"pga_a{alpha}_s{seed}.json"), "w") as f:
            json.dump(res, f, indent=2)
    f1s = [r['test']['macro_f1'] for r in pga_results if r['alpha']==alpha]
    print(f"  >>> PGA alpha={alpha}: F1={np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR, "pga_all.json"), "w") as f:
    json.dump(pga_results, f, indent=2)
print("\nAll PGA results saved.")

In [ ]:
# Comparison table: PGA vs FedAvg
print("\n" + "="*60)
print("PGA vs FedAvg COMPARISON (Bottleneck N/4)")
print("="*60)
print(f"{'alpha':<8} {'Method':<12} {'F1 mean':>9} {'F1 std':>8} {'Delta':>7} {'p-value':>9}")
print("-"*54)

ALPHAS = [1000, 1, 0.1]
SEEDS  = [0, 1, 2, 3, 4]

for alpha in ALPHAS:
    pga_f1s = [r['test']['macro_f1'] for r in pga_results if r['alpha']==alpha]
    try:
        fa_f1s = []
        for seed in SEEDS:
            path = os.path.join(P2_DIR, f"fedavg_bottleneck_r4_a{alpha}_s{seed}.json")
            with open(path) as f:
                fa_f1s.append(json.load(f)['test']['macro_f1'])
        _, pv    = ttest_rel(pga_f1s, fa_f1s)
        delta    = np.mean(pga_f1s) - np.mean(fa_f1s)
        print(f"{alpha:<8} {'FedAvg':<12} {np.mean(fa_f1s):>9.3f} {np.std(fa_f1s):>8.3f} {'---':>7} {'---':>9}")
        print(f"{alpha:<8} {'PGA':<12} {np.mean(pga_f1s):>9.3f} {np.std(pga_f1s):>8.3f} {delta:>+7.3f} {pv:>9.4f}")
    except FileNotFoundError:
        print(f"{alpha:<8} {'PGA':<12} {np.mean(pga_f1s):>9.3f} {np.std(pga_f1s):>8.3f} (P2 files not found)")
    print()

print("Files saved to:", RESULTS_DIR)
print("NEXT: Save output as 'feduca-phase3-results', then run Phase 4")